# 🎛️ 13 · Precise alignment, crowded neighbours and scaling up to millions
### Polishing coarse alignments with Levenberg–Marquardt, isolating the target in crowded boxes, and spreading the work over many processors

**BiologyToResearch** · Part C — Classical computational cryo-ET (2009–2019) · Module 13 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/13_precise_alignment_crowding_scale.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 4–5 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`optimize`, `ndimage`, `spatial.transform`), scikit-image, Matplotlib |
| **Prerequisites** | Modules 02 (rotations), 03 (Fourier), 04 (statistics), 07 (missing wedge), **11 (constrained correlation)** |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Xu & Alber 2012** (BMC Syst Biol — gradient-based high-precision alignment); **Xu & Alber 2013** (Bioinformatics/ISMB — target segmentation in crowded subtomograms); **Frazier, Xu & Alber 2017** (Structure — TomoMiner/TomoMinerCloud); **Lü *et al.* 2019** (BMC Bioinformatics — fine-grained alignment with MPI and stochastic average gradient) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments here are small, controlled *toy versions* of published methods. They illustrate the core ideas of the
> papers; they do **not** reproduce their numbers (we use 40³ boxes, a handful of particles, simplified noise and a
> simplified segmentation). **All molecules and data are generated by code** (nothing is downloaded): realistic
> stand-ins with the size and symmetry of the PDB entries they are named after.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Relate** angular alignment error to the blur it causes in an average, and say why coarse alignment is not enough.
2. **Write** alignment as a least-squares problem on missing-wedge-constrained volumes, and **show** that minimising the
   squared difference of normalised volumes is the same as maximising constrained correlation ($d = 2-2c$).
3. **Implement** Levenberg–Marquardt (LM) refinement of the 6 rigid-body parameters with a finite-difference Jacobian,
   and **compare** it with plain gradient descent and with SciPy's `least_squares`.
4. **Measure** the basin of attraction of local refinement, and **implement** stochastic refinement of many seeds with a
   shared budget (the idea of Xu & Alber 2012).
5. **Use** prior knowledge (a membrane normal) to shrink a 3D rotational search to a 1D one.
6. **Segment** the central target of a crowded subtomogram (smoothing + watershed, after Xu & Alber 2013) and **show**
   that masking out neighbours improves alignment.
7. **Scale** alignment with a master/worker pool (the TomoMiner pattern), estimate the cost of a real dataset, and
   **explain** MPI parallelism and stochastic average gradient (SAG) optimisation (Lü *et al.* 2019).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, molecules | 5 min |
| 1 · Why precision matters | angular error → blur at the edge of a molecule | 10 min |
| 2 · Alignment as least squares | residuals, $d=2-2c$, gradient descent vs Levenberg–Marquardt | 25 min |
| 3 · Many seeds, one budget | basin of attraction; stochastic parallel refinement | 20 min |
| 4 · Using what you know | membrane proteins: a 1D search instead of 3D | 10 min |
| 5 · Crowded neighbours | target segmentation and masked alignment | 25 min |
| 6 · Scaling up | master/worker pools, cost estimates, MPI and stochastic average gradient | 20 min |
| 📄 / ✅ / 🧠 / 🚀 | paper connections, checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

In Module 12 you aligned subtomograms quickly by scoring rotations on a grid. A 10° grid, however, leaves each copy up
to ~5° away from its true orientation — and for a ribosome 25 nm across, 5° at its rim means a ~1 nm smear, enough to
erase helices and RNA grooves in the average. Real cellular subtomograms add two more problems: the box around a ribosome
also contains pieces of its **neighbours** (other ribosomes, membranes, chaperones), which pull the alignment toward the
wrong answer; and there are **hundreds of thousands** of subtomograms to align against several references, many times
over. Between 2012 and 2019, a series of Alber-lab and Xu-lab papers attacked each of these problems: gradient-based
high-precision refinement, automatic target segmentation, and parallel software (TomoMiner, MPI) that turned the
methods into tools for whole-cell datasets.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("skimage", "scikit-image")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import concurrent.futures as cf
import multiprocessing as mp
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from scipy.optimize import least_squares
from scipy.spatial.transform import Rotation
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu
from skimage.segmentation import watershed

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The helpers below are copied from [`course_toolkit.py`](../course_toolkit.py) so this notebook runs on its own. You have
met all of them before: `rotate_volume` (Module 02), `missing_wedge_mask` (Module 07), `pearson` and `add_noise_snr`
(Module 04).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays) of the code-generated stand-in for zoo entry `pid` (Å, centred at the origin)."""
    pid = pid.lower()
    if pid in _ATOMS_CACHE:
        return _ATOMS_CACHE[pid]
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    _ATOMS_CACHE[pid] = atoms
    return atoms


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of the code-generated stand-in for a zoo entry (no download)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, f"code-generated stand-in modelled on PDB {pid.upper()}"


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def rotation_angle_deg(R1, R2) -> float:
    """Geodesic distance between two rotations: arccos((tr(R1ᵀR2) − 1)/2), in degrees."""
    c = (np.trace(R1.T @ R2) - 1) / 2
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes
print("🧰 toolkit loaded")

Two helpers from Module 12 complete the kit: the **rotated wedge mask** $(\Lambda_R M)(\xi)=M(R^{\mathsf T}\xi)$ — when we
rotate a subtomogram, its missing wedge rotates with it — and a function that **images** an object (rotate, shift, remove
the wedge, add noise).

In [ ]:
# ── Helpers from Module 12: rotated wedge masks and simulated observations ──
def wedge_mask_rotated(shape, R, tilt_max_deg=60.0):
    """Missing-wedge mask of a volume that was rotated by R after imaging: (Λ_R M)(ξ) = M(Rᵀξ). R acts on (x, y, z)."""
    k = freq_grid(shape)
    vec = np.stack([k[2], k[1], k[0]])                     # (x, y, z) components of every frequency
    kp = np.tensordot(R.T, vec, axes=1)                    # Rᵀξ
    m = np.abs(kp[2]) <= np.abs(kp[0]) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[0, 0, 0] = True
    return m.astype(np.float32)


def observe(obj, R, shift_xyz, snr, tilt_max_deg, rng):
    """Rotate/shift an object, then image it: missing wedge → white noise (SNR = var ratio) → missing wedge again.

    Returns (subtomogram, its Fourier mask). Applying the wedge after the noise mimics a reconstruction, in which the
    unmeasured frequencies are empty for signal AND noise.
    """
    x = rotate_volume(obj, R, shift_xyz)
    W = missing_wedge_mask(x.shape, tilt_max_deg)
    s = apply_fourier_mask(x, W)
    noise = rng.normal(0, np.sqrt(s.var() / snr), x.shape)
    return apply_fourier_mask(s + noise, W), W


print("helpers ready")

In [ ]:
# ── 0.1 The molecules of this module ──
N = 40                                                     # box size in voxels (1 voxel = 1 nm); big enough that
                                                           # a rotated ribosome never pokes out of the box
CAST = ["4v4r", "1qvr", "1bxn"]                            # ribosome (target in §2–4), ClpB (target in §5), RuBisCO (neighbour)
raw = {}
for pid in CAST:
    vol, vs, src = zoo_map(pid)
    raw[pid] = vol
    print(f"{pid.upper()} {ZOO[pid]['name']:24s} {vol.shape} @ {vs:.0f} Å/voxel ← {src}")
ribo = prepare_map(raw["4v4r"], size=N, sigma=1.0)
clpb = prepare_map(raw["1qvr"], size=N, sigma=1.0)
rubisco = prepare_map(raw["1bxn"], size=N, sigma=1.0)
show_orthoslices(ribo, "ribosome")
plt.show()

---
## 1 · Why precision matters

📍 **Where we are.** Module 12 gave us a *fast* aligner: score a grid of rotations with the phase-free score $c'$, keep
the best candidates, and find the shift with an FFT. Its answer is only as fine as the grid. This module turns a coarse
answer into a precise one, then deals with crowding and with scale.

A rotation by a small angle $\Delta\theta$ (in radians) moves a point at distance $r$ from the rotation centre by about

$$\delta \;\approx\; r\,\Delta\theta .$$

*In words:* the outer parts of a molecule move the most. If every copy in an average is misaligned by ~$\Delta\theta$,
details smaller than ~$2\delta$ at radius $r$ are blurred away. The next cell turns this into numbers.

In [ ]:
# ── 1.1 Angular error → displacement at the rim of a molecule ──
radii_nm = {"RuBisCO (r ≈ 5 nm)": 5, "20S proteasome (r ≈ 7.5 nm)": 7.5, "70S ribosome (r ≈ 12 nm)": 12}
angles_deg = np.linspace(0, 15, 100)
fig, ax = plt.subplots(figsize=(6, 3.4))
for lab, r in radii_nm.items():
    ax.plot(angles_deg, r * np.radians(angles_deg), label=lab)
ax.axhline(1.0, color="gray", ls="--", lw=1)
ax.text(0.3, 1.05, "1 nm (one voxel here)", fontsize=8, color="gray")
ax.axvline(5, color="tab:red", ls=":", lw=1)
ax.text(5.2, 2.6, "error left by a\n10° rotation grid", fontsize=8, color="tab:red")
ax.set_xlabel("angular alignment error Δθ (degrees)"); ax.set_ylabel("displacement at the rim δ (nm)")
ax.set_title("Small angles, large smears: δ ≈ r·Δθ"); ax.legend(fontsize=8)
plt.show()
for lab, r in radii_nm.items():
    print(f"{lab:30s}: 5° → {r * np.radians(5):.2f} nm;  1° → {r * np.radians(1):.2f} nm")

🔍 **What just happened?** At the ribosome's rim a 5° error moves density by ~1 nm — a full voxel. Sub-nanometre
averages need errors of ~1° or less. Scanning finer grids is hopeless: the number of rotations grows as
$1/\Delta\theta^3$ (about 8× more rotations each time the step is halved). Instead we **refine**: start from the coarse
answer and move *downhill* on the score, continuously.

---
## 2 · Alignment as a least-squares problem

### 2.1 Residuals and the identity $d = 2 - 2c$

Collect the six rigid-body parameters in a vector $\beta=(\omega_x,\omega_y,\omega_z,t_x,t_y,t_z)$: a small rotation
(as a rotation vector, in radians) and a shift (in voxels). For a subtomogram $g$ and a reference $f$:

1. transform $g$ by $\beta$ → $g_\beta$;
2. keep only Fourier coefficients measured in **both** volumes, $\Omega = M_f\cdot\Lambda_R M_g$ (the constraint of
   Module 12) → $\tilde f$, $\tilde g_\beta$;
   We also apply a soft **low-pass filter** at 1/(5 nm): at this noise level the finer frequencies are mostly noise and
   would make the score jagged (alignment codes routinely band-limit the data in the same way);
3. remove the means and scale each to unit length → $\hat f$, $\hat g_\beta$;
4. the **residual** is the voxel-wise difference $r(\beta)=\hat f-\hat g_\beta$.

The **dissimilarity** is the sum of squared residuals. Because both volumes have unit length,

$$d(\beta)=\lVert\hat f-\hat g_\beta\rVert^2=\lVert\hat f\rVert^2+\lVert\hat g_\beta\rVert^2-2\,\hat f\cdot\hat g_\beta = 2-2\,c(\beta),$$

where $c$ is exactly the constrained correlation of Module 12. *In words:* making the residual small **is** making the
correlation large — so we can borrow the powerful machinery of least squares. Xu & Alber (2012) call the real-space
version the *real-space constrained score*.

💡 We parameterise the rotation as a small rotation vector applied on top of the current estimate,
$R(\omega)=\exp([\omega]_\times)\,R_0$, instead of three Euler angles: near any $R_0$ this behaves smoothly in every
direction (no gimbal lock, Module 02). The paper used Euler angles; the optimisation idea is the same.

In [ ]:
# ── 2.1 Residual function ──
BAND = (1 / (1 + np.exp((radial_frequency((N, N, N)) - 0.2) / 0.02))).astype(np.float32)   # soft low-pass at 5 nm


def filt_norm(v, omega):
    """Keep the Fourier coefficients in Ω, remove the mean and scale to unit length."""
    x = np.real(np.fft.ifftn(np.fft.fftn(v) * omega))
    x = x - x.mean()
    return (x / (np.linalg.norm(x) + 1e-12)).astype(np.float32)


def transform_params(beta, R0, t0):
    """Apply a parameter step β = (ω, t) on top of the current pose (R0, t0). Returns (R, t)."""
    R = Rotation.from_rotvec(beta[:3]).as_matrix() @ R0
    return R, t0 + beta[3:]


def make_residual(f, Mf, g, tilt_g, R0, t0, target_mask=None):
    """Return r(β) = f̂ − ĝ_β for poses near (R0, t0). Ω is fixed at R0 while β varies (it changes only stepwise)."""
    omega = Mf * wedge_mask_rotated(f.shape, R0, tilt_g) * BAND
    f_hat = filt_norm(f, omega)

    def residual(beta):
        R, t = transform_params(beta, R0, t0)
        gb = g if target_mask is None else g * target_mask         # §5: optional mask that travels with g
        return (f_hat - filt_norm(rotate_volume(gb, R, t), omega)).ravel()
    return residual


def pose_errors(R, t, R_true, t_true):
    """Rotation error (degrees, geodesic) and shift error (voxels) of an estimated alignment pose."""
    return rotation_angle_deg(R, R_true), float(np.linalg.norm(np.asarray(t) - np.asarray(t_true)))


# A test pair: f = clean ribosome template (no wedge needed), g = rotated, shifted, wedged, noisy copy
TILT = 60
R_obj = random_rotation(rng)                         # how the copy sits in the cell
t_obj = np.array([1.5, -2.0, 1.0])                   # and where (voxels)
g, Mg = observe(ribo, R_obj, t_obj, snr=0.5, tilt_max_deg=TILT, rng=rng)
f, Mf = ribo.copy(), np.ones_like(ribo)              # a clean reference has no missing wedge
R_true, t_true = R_obj.T, -R_obj.T @ t_obj           # the pose that undoes (R_obj, t_obj)
check = make_residual(f, Mf, g, TILT, R_true, t_true)(np.zeros(6))
print(f"at the true pose: d = {np.sum(check**2):.3f}  →  c = 1 − d/2 = {1 - np.sum(check**2) / 2:.3f}")
omega_true = Mf * wedge_mask_rotated(f.shape, R_true, TILT) * BAND
print(f"constrained correlation computed directly: "
      f"{pearson(filt_norm(f, omega_true), filt_norm(rotate_volume(g, R_true, t_true), omega_true)):.3f}")
fig, axes = plt.subplots(2, 3, figsize=(7.5, 5))
show_orthoslices(f, "reference f:", axes=axes[0])
show_orthoslices(g, "subtomogram g:", axes=axes[1])
plt.tight_layout(); plt.show()

🔍 The two numbers agree: $d=2-2c$. The subtomogram (SNR 0.5, ±60° wedge — the noise level of the 2012 paper's tests)
is a rotated, shifted, noisy, elongated version of the reference. Even at the *true* pose the correlation is well below
1, because noise never correlates with the reference.

### 2.2 Three ways to go downhill

Let $J$ be the **Jacobian**: the $n_{\text{voxels}}\times 6$ matrix of how every residual changes when each parameter is
nudged. We estimate it by **finite differences** — change one parameter by a tiny $h$, recompute the residual, divide the
change by $h$ (6 extra residual evaluations per step).

| Method | Step $\Delta\beta$ | Behaviour |
|---|---|---|
| **Gradient descent** | $-\eta\,J^{\mathsf T}r$ | always downhill, but needs a good step size $\eta$; slow in long, narrow valleys |
| **Gauss–Newton** | $-(J^{\mathsf T}J)^{-1}J^{\mathsf T}r$ | fast near the minimum (uses curvature), can overshoot far from it |
| **Levenberg–Marquardt** | $-(J^{\mathsf T}J+\lambda\,\mathrm{diag}(J^{\mathsf T}J))^{-1}J^{\mathsf T}r$ | blends the two: large $\lambda$ → careful gradient-like steps; small $\lambda$ → fast Gauss–Newton steps |

LM adapts $\lambda$ itself: if a step lowers $d$, accept it and **decrease** $\lambda$ (be bolder); if not, reject it and
**increase** $\lambda$ (be more careful). This is the optimiser used by **Xu & Alber (2012)** — they derived a variant for
the complex-valued Fourier-space residuals as well; we use the real-space version.

In [ ]:
# ── 2.2 Finite-difference Jacobian, Levenberg–Marquardt and gradient descent ──
H = np.array([0.01, 0.01, 0.01, 0.1, 0.1, 0.1])              # finite-difference steps: 0.01 rad (0.57°), 0.1 voxel


def jacobian(residual, r0):
    """Finite-difference Jacobian (n × 6) of a residual function at β = 0."""
    J = np.empty((r0.size, 6), np.float32)
    for k in range(6):
        e = np.zeros(6); e[k] = H[k]
        J[:, k] = (residual(e) - r0) / H[k]
    return J


def lm_refine(f, Mf, g, tilt_g, R_init, t_init=(0, 0, 0), max_iter=25, lam=1e-2, tol=1e-3,
              truth=None, target_mask=None):
    """Levenberg–Marquardt refinement of a rigid pose. Returns (R, t, history list of dicts, n_residual_evaluations)."""
    R, t = np.array(R_init, float), np.array(t_init, float)
    hist, n_eval = [], 0
    for it in range(max_iter):
        res = make_residual(f, Mf, g, tilt_g, R, t, target_mask)
        r0 = res(np.zeros(6)); J = jacobian(res, r0); n_eval += 7
        d0 = float(r0 @ r0)
        A, b = J.T @ J, J.T @ r0
        for _ in range(6):                                   # adapt λ until the step helps
            step = -np.linalg.solve(A + lam * np.diag(np.diag(A)), b)
            r1 = res(step); n_eval += 1
            if r1 @ r1 < d0:
                lam = max(lam / 3, 1e-7)
                R, t = transform_params(step, R, t)
                break
            lam *= 4
        else:
            step = np.zeros(6)
        rec = dict(it=it, d=d0, lam=lam)                       # score at the start of the iteration (as for GD)
        if truth is not None:
            rec["rot_err"], rec["shift_err"] = pose_errors(R, t, *truth)
        hist.append(rec)
        if np.linalg.norm(step[:3]) < tol and np.linalg.norm(step[3:]) < 10 * tol:
            break
    return R, t, hist, n_eval


def gd_refine(f, Mf, g, tilt_g, R_init, t_init=(0, 0, 0), eta=0.05, n_iter=25, truth=None):
    """Plain gradient descent on d(β) with a fixed step size η (same Jacobian as LM)."""
    R, t = np.array(R_init, float), np.array(t_init, float)
    hist = []
    for it in range(n_iter):
        res = make_residual(f, Mf, g, tilt_g, R, t)
        r0 = res(np.zeros(6)); J = jacobian(res, r0)
        R, t = transform_params(-eta * (J.T @ r0), R, t)
        rec = dict(it=it, d=float(r0 @ r0))
        if truth is not None:
            rec["rot_err"], rec["shift_err"] = pose_errors(R, t, *truth)
        hist.append(rec)
    return R, t, hist


# start 15° away from the truth (about what a coarse grid leaves), and 2 voxels off
axis = rng.normal(size=3); axis /= np.linalg.norm(axis)
R_start = Rotation.from_rotvec(np.radians(15) * axis).as_matrix() @ R_true
t_start = t_true + np.array([2.0, 0.0, 0.0])
truth = (R_true, t_true)
print("start:   rotation error %.1f°, shift error %.1f voxels" % pose_errors(R_start, t_start, *truth))

t0 = time.time()
R_lm, t_lm, hist_lm, ne = lm_refine(f, Mf, g, TILT, R_start, t_start, truth=truth)
print(f"LM:      rotation error {hist_lm[-1]['rot_err']:.2f}°, shift error {hist_lm[-1]['shift_err']:.2f} voxels "
      f"after {len(hist_lm)} iterations ({ne} residual evaluations, {time.time() - t0:.1f} s)")
hist_gd = {}
for eta in (0.02, 0.1):
    t0 = time.time()
    *_, hist_gd[eta] = gd_refine(f, Mf, g, TILT, R_start, t_start, eta=eta, n_iter=25, truth=truth)
    print(f"GD η={eta:<4}: rotation error {hist_gd[eta][-1]['rot_err']:.2f}°, shift error "
          f"{hist_gd[eta][-1]['shift_err']:.2f} voxels after 25 iterations ({time.time() - t0:.1f} s)")

In [ ]:
# ── 2.3 Convergence plots ──
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
for lab, h, st in [("Levenberg–Marquardt", hist_lm, "o-")] + [(f"gradient descent η={e}", hist_gd[e], ".-") for e in hist_gd]:
    axes[0].plot([r["d"] for r in h], st, ms=3, label=lab)
    axes[1].plot([r["rot_err"] for r in h], st, ms=3, label=lab)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("dissimilarity d = 2 − 2c"); axes[0].set_title("score")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("rotation error (°)"); axes[1].set_yscale("log")
axes[1].set_title("distance to the true pose"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** LM roughly halves the rotation error (from 15° to below 10°, most of the gain in the first
few iterations) and cuts the shift error to a fraction of a voxel; SciPy's implementation (next cell) ends at a similar
pose. The remaining degrees are not the optimiser's fault: at SNR 0.5, a ±60° missing wedge and a 5 nm low-pass, our
roundish ribosome stand-in gives a score that is nearly flat within several degrees of the truth (the 2012 paper reported
mean errors of ~3° at SNR 0.5 on its own test data). Gradient descent with these step
sizes barely moves in the same number of iterations (a larger step makes it zig-zag or diverge). Every iteration costs
the same (7 residual evaluations for the Jacobian), so **iterations are the cost** — and LM wins because it uses the
curvature $J^{\mathsf T}J$.

✏️ **Your turn.** Re-run 2.2 with `snr=0.1` in cell 2.1 (harder data) or a 30° starting error (change
`np.radians(15)`). When does LM stop reaching the true pose?

In [ ]:
# ── 2.4 The same refinement with SciPy's least_squares (method="lm") ──
res0 = make_residual(f, Mf, g, TILT, R_start, t_start)
t0 = time.time()
sol = least_squares(res0, np.zeros(6), method="lm", diff_step=0.01, max_nfev=200)
R_sp, t_sp = transform_params(sol.x, R_start, t_start)
print(f"scipy least_squares: rotation error {pose_errors(R_sp, t_sp, *truth)[0]:.2f}°, shift error "
      f"{pose_errors(R_sp, t_sp, *truth)[1]:.2f} voxels, {sol.nfev} residual evaluations, {time.time() - t0:.1f} s")
print(f"our LM             : rotation error {hist_lm[-1]['rot_err']:.2f}°, shift error {hist_lm[-1]['shift_err']:.2f} voxels")

💡 SciPy's MINPACK implementation agrees with our 30-line version (it keeps $\Omega$ fixed at the starting pose, which is
fine for small corrections). In research code you would use a library; writing it once yourself shows there is no magic.

---
## 3 · Many seeds, one budget

### 3.1 The basin of attraction

Local refinement only finds the minimum of the valley it starts in. How far from the truth can we start and still
succeed? We start LM from poses $\Delta\theta$ away from the truth in random directions and count successes (final error
< 8°, i.e. down to the precision floor of this noisy toy data that we saw in 2.3).

In [ ]:
# ── 3.1 Success rate versus starting error ──
start_errors = [10, 30, 50, 70, 90]
n_trials = 5
success = {}
t0 = time.time()
for e0 in start_errors:
    ok = 0
    for k in range(n_trials):
        ax_ = rng.normal(size=3); ax_ /= np.linalg.norm(ax_)
        R0 = Rotation.from_rotvec(np.radians(e0) * ax_).as_matrix() @ R_true
        R_hat, t_hat, h, _ = lm_refine(f, Mf, g, TILT, R0, t_true, max_iter=15, truth=truth)
        ok += h[-1]["rot_err"] < 8
    success[e0] = ok / n_trials
    print(f"start {e0:3d}° away: {ok}/{n_trials} converge to < 8°")
print(f"({time.time() - t0:.0f} s)")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(list(success), list(success.values()), "o-")
ax.set_xlabel("starting rotation error (°)"); ax.set_ylabel("fraction converged (< 8°)")
ax.set_ylim(-0.05, 1.05); ax.set_title("Basin of attraction of LM refinement (ribosome, SNR 0.5)")
plt.show()

🔍 Refinement is reliable only from a reasonably close start; far away it gets trapped in another valley (a different
orientation of the blob that happens to match locally). The cure is to start from **several seeds** spread over all
orientations, refine them, and keep the best — but refining every seed to convergence wastes most of the effort on
seeds that were never going to win.

### 3.2 Stochastic parallel refinement

**Xu & Alber (2012)** proposed to treat the seeds as a *population* that shares one budget:

1. keep a list of candidate poses, each with its current score $d$;
2. repeatedly **pick one candidate at random, favouring better-ranked ones**, and give it **one** LM step;
3. **merge** candidates that converge to the same pose (keep the better);
4. stop when the leading candidate has converged.

Good candidates receive most steps; poor ones are rarely touched. ("Parallel" here means *many candidates refined
together*, not multiple processors.) We compare it with refining every seed independently to convergence. Our seeds are
the 24 rotations of the octahedral group, which cover rotation space with a spacing of ~60° — similar to the paper's
coarsest 60° seed sampling (they used 108 seeds).

In [ ]:
# ── 3.2 Independent refinement of every seed vs stochastic shared-budget refinement ──
def lm_step(f, Mf, g, tilt_g, R, t, lam):
    """One LM iteration from (R, t). Returns (R, t, d, λ, step_norm, n_residual_evaluations)."""
    res = make_residual(f, Mf, g, tilt_g, R, t)
    r0 = res(np.zeros(6)); J = jacobian(res, r0); n = 7
    d0 = float(r0 @ r0); A, b = J.T @ J, J.T @ r0
    for _ in range(4):
        step = -np.linalg.solve(A + lam * np.diag(np.diag(A)), b)
        r1 = res(step); n += 1
        if r1 @ r1 < d0:
            R, t = transform_params(step, R, t)
            return R, t, float(r1 @ r1), max(lam / 3, 1e-7), float(np.linalg.norm(step[:3])), n
        lam *= 4
    return R, t, d0, lam, 0.0, n


SEEDS = Rotation.create_group("O").as_matrix()              # 24 rotations, ~60° apart
t_seed = np.zeros(3)

# (a) independent: refine every seed until it stops moving
t0 = time.time()
indep, steps_indep = [], 0
for R0 in SEEDS:
    R_hat, t_hat, h, _ = lm_refine(f, Mf, g, TILT, R0, t_seed, max_iter=20, truth=truth)
    indep.append((h[-1]["d"], h[-1]["rot_err"])); steps_indep += len(h)
best_indep = min(indep)
time_indep = time.time() - t0
print(f"independent: {steps_indep} LM steps in total; best seed ends at d = {best_indep[0]:.3f}, "
      f"rotation error {best_indep[1]:.1f}°  ({time_indep:.0f} s)")


# (b) stochastic, shared budget
def stochastic_parallel_refine(seeds, budget=400, temperature=2.0, merge_deg=8.0, rng=rng):
    """Population refinement: rank-weighted random choice of the candidate that receives the next LM step."""
    cands = []
    for R0 in seeds:                                           # score every seed once (1 evaluation each)
        res = make_residual(f, Mf, g, TILT, R0, t_seed)
        r0 = res(np.zeros(6))
        cands.append(dict(R=R0, t=t_seed.copy(), d=float(r0 @ r0), lam=1e-2, moved=1.0, steps=0))
    trace = []
    queue = [c for c in cands for _ in range(2)]               # burn-in: two LM steps for every seed first
    for step in range(budget):
        if queue:
            c = queue.pop(0)
            if not any(c is k for k in cands):                 # this seed was merged away meanwhile
                continue
        else:                                                  # then: rank-weighted random choice
            order = np.argsort([c["d"] for c in cands])
            p = np.exp(-np.arange(len(cands)) / temperature); p /= p.sum()
            c = cands[order[rng.choice(len(cands), p=p)]]
        c["R"], c["t"], c["d"], c["lam"], c["moved"], _ = lm_step(f, Mf, g, TILT, c["R"], c["t"], c["lam"])
        c["steps"] += 1
        merged = []                                            # merge duplicates (same pose within merge_deg)
        for a in sorted(cands, key=lambda c: c["d"]):
            if all(rotation_angle_deg(a["R"], m["R"]) > merge_deg for m in merged):
                merged.append(a)
        cands = merged
        best = min(cands, key=lambda c: c["d"])
        trace.append((step + 1, best["d"], pose_errors(best["R"], best["t"], *truth)[0], len(cands)))
        if not queue and best["steps"] >= 4 and best["moved"] < 1e-3:   # the leader has converged
            break
    return best, trace


t0 = time.time()
best_sp, trace = stochastic_parallel_refine(SEEDS)
time_sp = time.time() - t0
print(f"stochastic:  {trace[-1][0]} LM steps in total; best candidate d = {best_sp['d']:.3f}, rotation error "
      f"{trace[-1][2]:.1f}°; {trace[-1][3]} candidates left after merging  ({time_sp:.0f} s)")
print(f"→ {steps_indep / trace[-1][0]:.1f}× fewer LM steps than refining every seed independently")
print(f"nearest seed to the true rotation: {min(rotation_angle_deg(S, R_true) for S in SEEDS):.0f}° away")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
tr = np.array(trace)
axes[0].plot(tr[:, 0], tr[:, 2], ".-"); axes[0].set_yscale("log")
axes[0].set_xlabel("LM steps used (shared budget)"); axes[0].set_ylabel("error of current best (°)")
axes[0].axhline(best_indep[1], color="gray", ls="--", lw=1); axes[0].set_title("stochastic refinement")
axes[1].plot(tr[:, 0], tr[:, 3], ".-", color="tab:green")
axes[1].set_xlabel("LM steps used"); axes[1].set_ylabel("candidates alive"); axes[1].set_title("merging duplicates")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Refining every seed independently spends most of its steps on seeds that end in wrong
valleys. The stochastic scheduler first gives every seed two LM steps (enough to see which valley it is heading for),
then spends its budget on the leading candidates, merges duplicates, and stops once the leader has converged —
reaching the same best pose with several times fewer LM steps (dashed line: the best independent result). Notice what
it *cannot* do: here the best pose is still tens of degrees from the truth, because even the nearest of the 24 seeds
started more than 30° away, where 3.1 found refinement unreliable. A scheduler saves work; it cannot make up for seeds that miss
the right valley — which is why the paper also tested denser seed sampling. Xu & Alber reported 200–300 iterations for stochastic parallel refinement versus ~6,480 for independent runs
— a 20–40× speed-up with 60° seed sampling and 84× with 45° sampling. Our toy population is much smaller, so the gain is
smaller, but the mechanism is the same.

✏️ **Your turn.** Use the 60 rotations of the icosahedral group as seeds (`Rotation.create_group("I")`, spacing ~45°).
Does the speed-up of the stochastic scheme grow, as in the paper?

---
## 4 · Using what you know: membrane proteins

For a protein embedded in a membrane we often know one direction already: the **membrane normal**, measured from the
segmented membrane around the particle (Module 32). Rotating the particle so that its normal points along $z$ leaves only
the **in-plane angle** $\varphi$ (and the shift) unknown — a 1D search instead of 3D. Xu & Alber (2012) describe this
"constrained search" mode for membrane complexes.

In [ ]:
# ── 4.1 How many rotations? Full 3D grid vs in-plane only ──
for step in (10, 5, 2):
    n3d = 8 * np.pi**2 / np.radians(step)**3            # volume of rotation space / volume per grid cell (approx.)
    n1d = 360 / step
    print(f"{step:2d}° sampling: full 3D ≈ {n3d:9,.0f} rotations  |  in-plane only: {n1d:4.0f}  "
          f"(×{n3d / n1d:,.0f} fewer)")

In [ ]:
# ── 4.2 A 1D search when the normal is known ──
# Pretend the ribosome's z axis is its "membrane normal" (e.g. a ribosome docked on the ER membrane).
phi_true = 137.0
tilt_part = Rotation.from_euler("y", 25, degrees=True).as_matrix()   # the membrane is tilted by 25° in the tomogram
R_obj2 = tilt_part @ rot_z(np.radians(phi_true))
g2, _ = observe(ribo, R_obj2, (0, 0, 0), snr=0.3, tilt_max_deg=TILT, rng=rng)
g2_upright = rotate_volume(g2, tilt_part.T)                         # step 1: put the measured normal along z
phis = np.arange(0, 360, 5.0)
scores = []
for phi in phis:                                                     # step 2: scan the in-plane angle only
    Rz_ = rot_z(np.radians(-phi))
    om = wedge_mask_rotated((N, N, N), Rz_ @ tilt_part.T, TILT) * BAND
    scores.append(pearson(filt_norm(ribo, om), filt_norm(rotate_volume(g2_upright, Rz_), om)))
phi_hat = phis[int(np.argmax(scores))]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(phis, scores); ax.axvline(phi_true, color="tab:red", ls="--", label=f"true φ = {phi_true:.0f}°")
ax.set_xlabel("in-plane angle φ (°)"); ax.set_ylabel("constrained correlation"); ax.legend(fontsize=8)
ax.set_title(f"72 rotations instead of thousands: best φ = {phi_hat:.0f}°")
plt.show()

🔍 With the normal known, 72 evaluations find the in-plane angle to within the 5° step (and LM can polish it). The price:
the result is only as good as the normal estimate — a 10° error in the membrane segmentation becomes a 10° alignment
error that the 1D search cannot fix.

---
## 5 · Crowded neighbours: segment the target first

Inside a cell a 40 nm box around a molecule almost always contains parts of its neighbours. When we rotate the box to
match a reference, the neighbours rotate too, and their density pulls the score toward poses that match *them*.

**Xu & Alber (2013)** proposed to **segment the central target automatically** before aligning: smooth the subtomogram,
find candidate structure boundaries, split the box into regions with a **watershed** seeded at density maxima, keep the
region at the centre, and dilate it into a mask. They also introduced a fast real-space alignment that operates on the
masked target. Our simplified version:

1. Gaussian smoothing (σ = 1.5 voxels) — noise reduction for segmentation only;
2. foreground = smoothed density above an automatic (Otsu) threshold;
3. **watershed** of the inverted smoothed density, seeded at local maxima, inside the foreground;
4. keep the segment containing the box centre (or the nearest one); dilate by 2 voxels; soften the edge.

(The paper's segmentation is more elaborate: it selects the smoothing scale automatically, detects boundary voxels from
the dark shell that the CTF creates around complexes, and clusters segments with a Gaussian mixture model.)

In [ ]:
# ── 5.1 Crowded subtomograms: a ClpB target with ribosome and RuBisCO neighbours ──
def crowded_box(rng, n_neighbours=3, dist=(18, 21)):
    """A 40³ box: rotated ClpB at the centre + one ribosome (≈ 25 voxels away, partly inside the box) and RuBisCOs
    touching the target (centres 18–21 voxels away). Returns (box, target, neighbours, R)."""
    R = random_rotation(rng)
    target = rotate_volume(clpb, R)
    neigh = np.zeros_like(target)
    for k in range(n_neighbours):
        tmpl, d = (ribo, 25.0) if k == 0 else (rubisco, rng.uniform(*dist))
        v = rng.normal(size=3); v /= np.linalg.norm(v)
        neigh += rotate_volume(tmpl, random_rotation(rng), v * d)
    return target + neigh, target, neigh, R


def segment_target(sub, sigma=1.5, dilate=2):
    """Simplified target segmentation in the spirit of Xu & Alber (2013): smooth → Otsu → watershed → central segment."""
    sm = ndimage.gaussian_filter(sub, sigma)
    fg = sm > threshold_otsu(sm)
    peaks = peak_local_max(sm, min_distance=4, labels=fg.astype(int))
    markers = np.zeros(sub.shape, int)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    labels = watershed(-sm, markers, mask=fg)
    c = np.array(sub.shape) // 2
    lab = labels[tuple(c)]
    if lab == 0:                                                    # centre fell outside: take the nearest segment
        pts = np.argwhere(labels > 0)
        lab = labels[tuple(pts[np.argmin(np.linalg.norm(pts - c, axis=1))])]
    mask = ndimage.binary_dilation(labels == lab, iterations=dilate)
    return ndimage.gaussian_filter(mask.astype(np.float32), 1.0), labels


box, target, neigh, R_t = crowded_box(np.random.default_rng(7))
noisy, Mbox = observe(box, np.eye(3), (0, 0, 0), snr=0.5, tilt_max_deg=TILT, rng=rng)
mask, labels = segment_target(noisy)
tgt_support = target > 0.2 * target.max()
nb_support = neigh > 0.2 * neigh.max()
print(f"target voxels covered by the mask: {mask[tgt_support].mean():.2f}   |   "
      f"neighbour voxels inside the mask: {mask[nb_support].mean():.3f}")
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
z = N // 2
for ax, im, ttl in zip(axes, [box[z], noisy[z], labels[z], (noisy * mask)[z]],
                       ["true density (target + neighbours)", "noisy subtomogram", "watershed segments", "masked target"]):
    ax.imshow(im, cmap="nipy_spectral" if "segments" in ttl else "gray"); ax.set_title(ttl, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 The watershed splits the box into blobs; the one at the centre is our ClpB. The mask keeps most of the target and very
little of the neighbours (Xu & Alber reported a median target overlap ≥ 0.9 and a median overlap with surrounding
structures ≤ 0.004 on their simulated crowded data).

### 5.2 Does masking improve alignment?

We now align 8 crowded subtomograms to the clean ClpB reference with LM, starting 20° from the truth, **with** and
**without** the target mask. The mask is computed once on the subtomogram and then rotated *together with it*.

In [ ]:
# ── 5.2 Unmasked vs masked refinement of crowded subtomograms ──
t0 = time.time()
errs = {"unmasked": [], "masked": []}
f_ref, Mf_ref = clpb.copy(), np.ones_like(clpb)
for k in range(8):
    box_k, target_k, neigh_k, R_k = crowded_box(np.random.default_rng(100 + k))
    g_k, _ = observe(box_k, np.eye(3), (0, 0, 0), snr=0.5, tilt_max_deg=TILT, rng=rng)
    m_k, _ = segment_target(g_k)
    R_true_k = R_k.T
    ax_ = rng.normal(size=3); ax_ /= np.linalg.norm(ax_)
    R0 = Rotation.from_rotvec(np.radians(20) * ax_).as_matrix() @ R_true_k
    for name, tm in [("unmasked", None), ("masked", m_k)]:
        R_hat, t_hat, h, _ = lm_refine(f_ref, Mf_ref, g_k, TILT, R0, (0, 0, 0), max_iter=12,
                                       truth=(R_true_k, np.zeros(3)), target_mask=tm)
        errs[name].append(h[-1]["rot_err"])
print(f"({time.time() - t0:.0f} s)")
for name, e in errs.items():
    e = np.array(e)
    print(f"{name:9s}: median rotation error {np.median(e):5.1f}°, success (< 8°) {np.mean(e < 8):.0%}  "
          f"| per box: {np.round(e, 1)}")
fig, ax = plt.subplots(figsize=(5.2, 3))
ax.plot(errs["unmasked"], errs["masked"], "o")
lim = max(max(errs["unmasked"]), max(errs["masked"])) * 1.1
ax.plot([0.3, lim], [0.3, lim], "k--", lw=1); ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("error without mask (°)"); ax.set_ylabel("error with target mask (°)")
ax.set_title("Each point = one crowded subtomogram\n(below the diagonal = mask helps)")
plt.show()

🔍 **What just happened?** Points below the diagonal are boxes where segmenting the target first gave a better
alignment. Without the mask, the neighbours' density is part of what LM tries to match; since the reference contains no
neighbours, it can only be matched by a wrong rotation. In the paper, the earlier reciprocal-space alignment method
generally failed on crowded simulated subtomograms while the new masked, real-space method was only marginally affected;
on experimental *Leptospira interrogans* data, 18 of 20 subtomograms were segmented successfully (failures mainly above
50 % crowdedness).

✏️ **Your turn.** Increase the crowding (`n_neighbours=5`, `dist=(13, 16)` in `crowded_box`) and re-run 5.1–5.2. When
does the segmentation itself start to fail?

---
## 6 · Scaling up to millions of subtomograms

### 6.1 How much work is a real study?

In [ ]:
# ── 6.1 A back-of-the-envelope cost estimate ──
t0 = time.time()
_, _, h, n_ev = lm_refine(f, Mf, g, TILT, R_start, t_start, max_iter=8)
t_one = time.time() - t0
print(f"one LM refinement here ({N}³ box, {len(h)} iterations, {n_ev} residual evaluations): {t_one:.2f} s")
for n_sub, n_ref, n_iter in [(10_000, 5, 10), (200_000, 10, 10), (1_000_000, 20, 15)]:
    cpu_h = n_sub * n_ref * n_iter * t_one / 3600
    print(f"{n_sub:>9,} subtomograms × {n_ref:2d} references × {n_iter:2d} rounds → {cpu_h:12,.0f} CPU-hours "
          f"= {cpu_h / 24:9,.0f} CPU-days on one core")

🔍 Even with fast code, a whole-cell study needs **thousands of CPU-hours** — which is only practical because the work is
**embarrassingly parallel**: aligning subtomogram A does not depend on aligning subtomogram B.

### 6.2 The master/worker pattern (TomoMiner)

**TomoMiner** (Frazier, Xu & Alber 2017) packages fast alignment, averaging, classification and pattern mining with a
task-queue architecture: a **queue server** holds tasks, a **master** submits chunks of work (e.g. "align these 500
subtomograms to these 3 references") and collects results, and many **workers** on a cluster — or in the cloud with
*TomoMinerCloud* — pull tasks and run them; unfinished tasks are resubmitted. Python's `concurrent.futures` gives us the
same pattern on one machine.

In [ ]:
# ── 6.2 Serial vs a worker pool ──
jobs = []
for k in range(8):
    Rk = random_rotation(rng)
    gk, _ = observe(ribo, Rk, (0, 0, 0), snr=0.5, tilt_max_deg=TILT, rng=rng)
    ax_ = rng.normal(size=3); ax_ /= np.linalg.norm(ax_)
    jobs.append((gk, Rotation.from_rotvec(np.radians(10) * ax_).as_matrix() @ Rk.T, Rk.T))


def run_job(job):
    """Worker task: refine one subtomogram against the reference; return its final rotation error."""
    gk, R0, Rt = job
    R_hat, t_hat, h, _ = lm_refine(f, Mf, gk, TILT, R0, (0, 0, 0), max_iter=6)
    return rotation_angle_deg(R_hat, Rt)


t0 = time.time(); serial = [run_job(j) for j in jobs]; t_serial = time.time() - t0
n_workers = min(4, os.cpu_count() or 1)
try:
    pool = cf.ProcessPoolExecutor(max_workers=n_workers, mp_context=mp.get_context("fork"))
    kind = "processes"
except (ValueError, RuntimeError):                          # 'fork' is unavailable on Windows/macOS-spawn
    pool = cf.ThreadPoolExecutor(max_workers=n_workers)
    kind = "threads"
t0 = time.time()
with pool:
    parallel = list(pool.map(run_job, jobs))
t_par = time.time() - t0
print(f"serial: {t_serial:.1f} s | {n_workers} worker {kind}: {t_par:.1f} s | speed-up ×{t_serial / t_par:.2f} "
      f"(ideal ×{n_workers}) on {os.cpu_count()} CPU cores")
assert np.allclose(serial, parallel), "workers must give exactly the same answers as the serial loop"
print("final errors (°):", np.round(parallel, 1))

🔍 The pool gives identical answers. Whether it is faster depends on the machine: each alignment already uses
multi-threaded NumPy/FFT code, and starting workers and moving data costs time, so on a 2-core machine such as Colab's
free tier the gain is at most ×2 and can vanish entirely (compare the printed speed-up with the ideal). A cluster node
with 64 cores, each working on its own chunk of subtomograms, gives far more. Speed-up is never perfect: the serial part
and the overheads limit it (**Amdahl's law**).

### 6.3 Parallelism inside one alignment: MPI and stochastic average gradient

**Lü *et al.* (2019)** sped up *fine-grained* alignment in two ways:

- **MPI** (Message Passing Interface — the standard for programs spread over many cluster nodes) evaluates the different
  candidate starting poses from fast alignment in parallel and keeps the best;
- each local refinement uses a **stochastic average gradient (SAG)** optimiser on the missing-wedge-constrained
  dissimilarity $d=2-2\,\mathrm{CCC}$: the volume is split into slices; every step computes the gradient of only a small
  batch of slices, stores it, and moves the parameters along the **average of the most recent gradients of all slices**:

$$\beta^{k}=\beta^{k-1}-\frac{\alpha}{n}\sum_{i=1}^{n}y_i^{k},$$

where $y_i^k$ is the last gradient computed for slice $i$. *In words:* each step is cheap (one batch of slices), but it is
guided by information from the whole volume. They reported reaching comparable or better resolution with ~3.7× less run
time than the earlier high-precision methods on 20 simulated GroEL subtomograms.

✏️ **Think it through.** Our LM refinement in Section 2 computes the residual over all $40^3$ voxels at every step. If
a step used only 4 of the 40 z-slices, it would cost 10× less — but would point in a noisier direction. How does SAG's
"memory" of every slice's last gradient fix that? What could go wrong if the memory is very old (hint: the parameters
have moved since those gradients were computed — SAG needs a small, carefully chosen step size)?

---
### 📄 Paper connection — Xu & Alber (2012): high-precision alignment by gradient-based parallel optimisation

**Xu M, Alber F.** *High precision alignment of cryo-electron subtomograms through gradient-based parallel optimization.*
BMC Systems Biology 6(Suppl 1):S18 (2012). [doi:10.1186/1752-0509-6-S1-S18](https://doi.org/10.1186/1752-0509-6-S1-S18).
A shorter conference version appeared as *Gradient-based high precision alignment of cryo-electron subtomograms*, IEEE
ISB 2011.
- **Question:** how to turn coarse alignments into sub-degree, sub-voxel ones without exhaustive fine scanning, robustly
  from few seeds.
- **What the paper does:** minimises a missing-wedge-constrained dissimilarity — in real space (RCS) or Fourier space
  (FCS) — over ZYZ Euler angles and translation with **Levenberg–Marquardt** (deriving a complex-residual variant for FCS;
  numerical Jacobians for rotations, analytic ones for translations in FCS), and refines many seeds jointly with a
  **stochastic parallel** scheduler that merges candidates converging to the same transform. It also offers a constrained
  search for membrane complexes (normal along z, search φ only).
- **What our toy keeps:** constrained real-space residual, $d = 2-2c$, LM with adaptive λ and finite differences,
  rank-weighted stochastic scheduling with merging, the membrane-normal 1D search.
- **What it leaves out:** the Fourier-space score and complex-residual LM, Euler-angle parameterisation, the
  linear-regression convergence test, the paper's seed counts and noise model.
- **Result reported:** at SNR 0.5 and ±70°, seeding with 60° sampling (108 seeds, versus 168,634 orientations for an
  exhaustive 5° scan) gave mean angular errors of 3.1° (RCS) or 2.9° (FCS); stochastic parallel refinement converged in
  200–300 iterations versus ~6,480 for independent runs (20–40× faster at 60° sampling, 84× at 45°).
- **Code:** the paper's implementation was MATLAB; AITom's `tomominer/align/refine/gradient_refine.py` placeholder is empty,
  so there is no runnable public version.

### 📄 Paper connection — Xu & Alber (2013): target segmentation and real-space fast alignment for crowded subtomograms

**Xu M, Alber F.** *Automated target segmentation and real space fast alignment methods for high-throughput
classification and averaging of crowded cryo-electron subtomograms.* Bioinformatics 29(13):i274–i282 (ISMB 2013).
[doi:10.1093/bioinformatics/btt225](https://doi.org/10.1093/bioinformatics/btt225).
- **Question:** subtomograms from crowded cells contain neighbours; how to isolate the target and align it robustly?
- **What the paper does:** a template-free **target segmentation** (automatic smoothing-scale selection; boundary voxels
  from negative-intensity minima — the dark CTF shell around complexes — scored by Hessian-based surface measures;
  watershed; k-means on boundary segments; an intensity-weighted Gaussian mixture with BIC to group maxima; the central
  cluster, dilated, is the mask) and a **real-space fast alignment** on the masked target (translate the constrained
  centres of mass together, then a spherical-harmonics FFT rotational search on real-space density).
- **What our toy keeps:** smoothing + watershed from maxima + central segment + dilation, and masking before alignment.
- **What it leaves out:** scale selection, boundary detection, the GMM step and the fast real-space rotational search.
- **Result reported:** median target overlap ≥ 0.9 and median overlap with surrounding structures ≤ 0.004 on simulated
  crowded data; 18/20 experimental *Leptospira* subtomograms segmented successfully; ~14.8 s per subtomogram. Note: despite
  the title, the paper does not report a classification or averaging experiment.
- **Code:** related routines in AITom `tomominer/segmentation/` and `tomominer/pursuit/multi/util.py`.

### 📄 Paper connection — Frazier, Xu & Alber (2017): TomoMiner and TomoMinerCloud

**Frazier Z, Xu M, Alber F.** *TomoMiner and TomoMinerCloud: a software platform for large-scale subtomogram structural
analysis.* Structure 25(6):951–961.e2 (2017). [doi:10.1016/j.str.2017.04.016](https://doi.org/10.1016/j.str.2017.04.016).
- **What it is:** the software platform that runs the lab's fast alignment, averaging, classification and Multi-Pattern
  Pursuit at scale: a C++ core (FFTW, spherical harmonics, rotational search) wrapped in Python, with a queue-server /
  master / worker architecture for clusters and a cloud-deployable version.
- ⚠️ We could not access the full text while writing; this description comes from the released code (alberlab/tomominer
  and AITom's `tomominer/` copy). Read the paper for its benchmarks and cloud set-up.

### 📄 Paper connection — Lü *et al.* (2019): fine-grained alignment with MPI parallel optimisation

**Lü Y, Zeng X, Zhao X, Li S, Li H, Gao X, Xu M.** *Fine-grained alignment of cryo-electron subtomograms based on MPI
parallel optimization.* BMC Bioinformatics 20:443 (2019). [doi:10.1186/s12859-019-3003-2](https://doi.org/10.1186/s12859-019-3003-2).
- **What the paper does:** refines candidate poses from fast alignment by minimising $d = 2-2\,\mathrm{CCC}$ with a 3D
  **stochastic average gradient** over slices (optimised batches of 1/16–1/2 of the side length), and evaluates candidate
  starting sets in parallel with **MPI**.
- **Result reported:** on simulated GroEL (SNR 0.003, ±60°) 37.1 Å in 14 iterations versus 40.7 Å (Xu 2012) and 39.7 Å
  (Chen *et al.*); on experimental GroEL 25.1 Å versus 32.5 Å and 27.9 Å; ~3.7× faster on 20 subtomograms.
- **What our toy keeps / leaves out:** keeps the parallel evaluation of independent tasks (Section 6.2) and explains the
  SAG update; we do not implement SAG (with only a few dozen slices and our simple step-size rule it was unstable — the
  paper's $\alpha = 1/L$ step rule matters), MPI itself, or the GroEL benchmark.

## ✅ Checkpoint

1. A 20S proteasome copy is misaligned by 3°. How far does density move at its ends (radius ≈ 7.5 nm)?
   <details><summary>Answer</summary>δ ≈ r·Δθ = 7.5 nm × 0.052 rad ≈ 0.39 nm.</details>
2. Why does minimising $\lVert\hat f-\hat g\rVert^2$ maximise the correlation?
   <details><summary>Answer</summary>For zero-mean, unit-length volumes, ‖f̂ − ĝ‖² = 2 − 2 f̂·ĝ, and f̂·ĝ is the (constrained) correlation.</details>
3. What does λ do in Levenberg–Marquardt?
   <details><summary>Answer</summary>It blends gradient descent (large λ: short, safe steps) with Gauss–Newton (small λ: fast, curvature-aware steps). LM lowers λ after a successful step and raises it after a failed one.</details>
4. Why refine many seeds, and why not refine all of them to convergence?
   <details><summary>Answer</summary>Local refinement only reaches the minimum of the valley it starts in, so seeds must cover all orientations; but most seeds end in wrong valleys, so a shared, rank-weighted budget wastes less effort on them.</details>
5. Why does a neighbouring ribosome inside the box bias alignment, and how does target segmentation help?
   <details><summary>Answer</summary>The neighbour rotates with the box and contributes to the score, which a wrong pose can increase; masking the central target removes that density before scoring.</details>
6. Why is subtomogram alignment called "embarrassingly parallel"?
   <details><summary>Answer</summary>Each subtomogram–reference alignment is independent of the others, so tasks can be distributed to workers without communication.</details>

## 🧠 Questions & discussion

1. LM needs a Jacobian. Which part of our residual is not smooth, and how did we handle it? (Hint: the rotated wedge mask.)
2. In Section 4, what happens if the membrane normal is 10° off? Design an experiment to measure it.
3. Masking removes neighbours — but some biology *is* in the neighbours (e.g. polysomes: ribosomes in chains). How would
   you study neighbour arrangements without letting them bias alignment?
4. Amdahl's law: if 5 % of a pipeline is serial, what is the maximum speed-up on 1,000 cores?
5. Fast rotational search (Module 12) and local refinement (this module) are complementary. Sketch a pipeline that uses
   both, and say where each spends its time.

## 🚀 Bonus — going further

- **Fourier-space score.** Rewrite the residual in Fourier space over Ω (complex numbers!) and compare its convergence with
  the real-space version, as the 2012 paper did.
- **Analytic translations.** In Fourier space a shift is a phase factor, so $\partial/\partial t$ is analytic. Replace the
  three translational finite differences and measure the saving.
- **Implement SAG.** Split the residual of Section 2 into z-slices, store each slice's last gradient, and compare loss
  versus work with full-batch gradient descent. Start with a small step size and increase it carefully.
- **Real software.** Look at AITom's `align/fast/` and `tomominer/` directories, and the
  [alberlab/tomominer](https://github.com/alberlab/tomominer) repository's queue server/master/worker code.

## 📝 Lab note

```text
Module 13 — date:
Input: clean reference + rotated/shifted/wedged/noisy ribosome and ClpB subtomograms; crowded boxes
Output: refined poses, basin of attraction, seed-refinement cost, masks, speed-ups
Method: Levenberg–Marquardt on the constrained residual; stochastic shared-budget refinement; watershed target masks;
        worker pools
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Xu & Alber 2012; Xu & Alber 2013; Frazier et al. 2017; Lü et al. 2019
```

## 📚 References

1. Xu M, Alber F. High precision alignment of cryo-electron subtomograms through gradient-based parallel optimization. *BMC Syst Biol* 6(Suppl 1):S18 (2012). doi:10.1186/1752-0509-6-S1-S18
2. Xu M, Alber F. Automated target segmentation and real space fast alignment methods for high-throughput classification and averaging of crowded cryo-electron subtomograms. *Bioinformatics* 29:i274–i282 (2013). doi:10.1093/bioinformatics/btt225
3. Frazier Z, Xu M, Alber F. TomoMiner and TomoMinerCloud: a software platform for large-scale subtomogram structural analysis. *Structure* 25:951–961 (2017). doi:10.1016/j.str.2017.04.016
4. Lü Y, Zeng X, Zhao X, Li S, Li H, Gao X, Xu M. Fine-grained alignment of cryo-electron subtomograms based on MPI parallel optimization. *BMC Bioinformatics* 20:443 (2019). doi:10.1186/s12859-019-3003-2
5. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012). doi:10.1016/j.jsb.2012.02.014
6. Moré JJ. The Levenberg–Marquardt algorithm: implementation and theory. In *Numerical Analysis*, Lecture Notes in Mathematics 630, pp. 105–116 (1978).
7. Schmidt M, Le Roux N, Bach F. Minimizing finite sums with the stochastic average gradient. *Math Program* 162:83–112 (2017).

## ➡️ Next

**[14 · Averaging and classification: reference-free iteration and maximum likelihood](14_averaging_classification_ml.ipynb)** —
once copies are aligned, average them, sort mixtures of molecules, and see how maximum likelihood copes with noise that
defeats "best-pose" alignment. Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).